# GPT-2 from Scratch — WikiText-2 학습 (Colab T4 GPU)

이 노트북은 GitHub 저장소의 코드를 Colab 환경에 클론한 뒤,
WikiText-2 데이터셋으로 GPT-2(축소판)를 직접 학습시키는 전체 과정을 담고 있습니다.

**실행 전 체크리스트**
- 런타임 유형: GPU (T4) 로 설정되어 있어야 합니다.
  (메뉴 > 런타임 > 런타임 유형 변경 > T4 GPU)


## 0. GPU 확인

In [ ]:
import torch

if torch.cuda.is_available():
    print("CUDA is available. You can use the GPU!")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Using the CPU instead.")


## 1. 저장소 클론 및 의존성 설치

`<YOUR_GITHUB_URL>` 부분을 실제 저장소 주소로 교체하세요.

In [ ]:
!git clone <YOUR_GITHUB_URL> gpt2-from-scratch
%cd gpt2-from-scratch
!pip install -q -r requirements.txt


## 2. 데이터 준비 (WikiText-2)

tiny Shakespeare 대신 **WikiText-2-raw** 데이터셋을 사용합니다.
HuggingFace `datasets`로 다운로드하고, GPT-2 원본 BPE 토크나이저(`tiktoken`)로 인코딩하여
`data/wikitext/{train,val}.bin` 으로 저장합니다.

In [ ]:
!python data/prepare_wikitext.py

## 3. 모델/학습 설정 확인 및 (선택) 수정

Colab 무료 T4(16GB) 기준 기본값이 이미 설정되어 있습니다.
빠른 데모 실행을 원하면 `TrainConfig.max_steps`를 줄이세요 (예: 500).

In [ ]:
from config import GPTConfig, TrainConfig

gpt_cfg = GPTConfig()
train_cfg = TrainConfig()

print(gpt_cfg)
print(train_cfg)

# 빠르게 결과를 확인하고 싶다면 아래 주석을 해제하세요.
# train_cfg.max_steps = 500
# train_cfg.eval_interval = 100


## 4. 학습 실행

전체 학습은 T4 기준 약 30분~1시간 정도 소요됩니다 (max_steps=3000 기준).

In [ ]:
!python train.py

## 5. 학습 로그 시각화

In [ ]:
import json
import matplotlib.pyplot as plt

with open("results/training_log.json") as f:
    history = json.load(f)

steps = [h["step"] for h in history]
train_losses = [h["train"] for h in history]
val_losses = [h["val"] for h in history]
val_ppls = [h["val_perplexity"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(steps, train_losses, label="train loss")
axes[0].plot(steps, val_losses, label="val loss")
axes[0].set_xlabel("step")
axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Training / Validation Loss")
axes[0].legend()

axes[1].plot(steps, val_ppls, color="tab:orange", label="val perplexity")
axes[1].set_xlabel("step")
axes[1].set_ylabel("perplexity")
axes[1].set_title("Validation Perplexity")
axes[1].legend()

plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=150)
plt.show()


## 6. 최종 Perplexity 평가 (validation set)

In [ ]:
!python evaluate.py --ckpt checkpoints/best_model.pt --split val

## 7. 텍스트 생성 결과 확인

WikiText로 학습된 모델이 어떤 스타일의 텍스트를 생성하는지 확인합니다.
(위키백과 문체 — 인물/지역/역사적 사실 서술 형태가 나오는지 살펴보세요.)

In [ ]:
!python generate.py \
    --ckpt checkpoints/best_model.pt \
    --prompt "The history of the United States" \
    --max_new_tokens 200 \
    --temperature 0.8 \
    --top_k 50


## 8. (참고) tiny Shakespeare 대비 WikiText 학습의 차이점

| 항목 | tiny Shakespeare | WikiText-2 (본 프로젝트) |
|---|---|---|
| 도메인 | 희곡 대사체 | 위키백과 백과사전 문체 |
| 어휘 다양성 | 제한적 (등장인물명, 고어체 표현 반복) | 인물/지명/숫자/전문용어 등 매우 다양 |
| 토큰화 | 보통 글자 단위(char-level) | BPE(subword) 단위, vocab_size=50,257 |
| 데이터 크기 | 약 1MB | 학습 약 2M 토큰 이상 (raw 기준) |
| 기대되는 결과물 | 셰익스피어풍 대사 생성 | 백과사전 서술형 문장 생성 |

WikiText는 문장 구조가 훨씬 다양하고 사실 기반 서술이 많기 때문에,
같은 모델 크기/학습 스텝 기준으로 perplexity가 tiny Shakespeare보다 높게 나오는 것이 일반적입니다
(이는 데이터 자체의 엔트로피가 더 높기 때문이며, 모델이 더 못 배운 것이 아닙니다).